<a href="https://colab.research.google.com/github/ClintonJuma/Deep-Mind/blob/main/NGrams_vs_Tranformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install -U protobuf
!pip install jax[cuda12]==0.7.2
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

#importing packages
import os
import pandas as pd

#Function for clearing outputs and fomratting
from IPython.display import clear_output, display, HTML

# Functions for generating texts with a language model, visualizing probability
# distributions, and loading an n-gram model.

from ai_foundations import generation#imports gemma-1B
from ai_foundations import visualizations
from ai_foundations.ngram import model as ngram_model#import NGRam model

# Set the full GPU memory usage for JAX.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"


ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

Loading Models


In [2]:
#importing dataset
africa_galore = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
)
africa_galore.head(1)

,category,name,description
0,Music,Afrobeat,"The Lagos air was thick with humidity, but the..."


In [3]:
dataset = africa_galore["description"]
dataset.head(1)

,description
0,"The Lagos air was thick with humidity, but the..."


In [4]:
#Loading a trigram model whose probabilities have been estimated using the dataset
trigram_model = ngram_model.NGramModel(dataset, 3)
print("Loaded trigram model.\n")

Loaded trigram model.



In [5]:
print("Loading Gemma-1B model...")
gemma_model = generation.load_gemma()
print("Loaded Gemma-1B model.")

Loading Gemma-1B model...
Loaded Gemma-1B model.


Comparing
 Model OutPuts

In [25]:
prompt="Juma was feeling sick so he went away looking for  "  # @param {type: "string"}

output_text_transformer, _, _=(
    generation.prompt_transformer_model(
        prompt, max_new_tokens = 4, loaded_model=gemma_model
    )
)
clear_output()
print(f"Generation by Gemma-1B:\n{output_text_transformer}\n\n")

output_text_ngram = trigram_model.generate(4, prompt)
print(f"Generation by trigram model:\n{output_text_ngram}")

Generation by Gemma-1B:
Juma was feeling sick so he went away looking for  a nurse.



⚠️ No valid continuation found. Change the prompt or try sampling another continuation.

Generation by trigram model:
Juma was feeling sick so he went away looking for   


In [39]:
# @title Visualize the probability distributions

prompt = "Jide was hungry so she went looking for"  # @param {type: "string"}

output_text_transformer, next_token_logits, tokenizer = (
    generation.prompt_transformer_model(
        prompt, max_new_tokens=1, loaded_model=gemma_model
    )
)

display(HTML("<h3>Gemma-1B</h3>"))

# Visualize the Gemma-1B probabilities.
visualizations.plot_next_token(
    next_token_logits,
    prompt=prompt,
    tokenizer=tokenizer
)

display(HTML("<h3>Trigram model</h3>"))

# Visualize the trigram probabilities.
context_ngram = tuple(prompt.split(" ")[-2:])
if context_ngram in trigram_model.probabilities:
    visualizations.plot_next_token(
        trigram_model.probabilities[context_ngram], prompt=prompt
    )
else:
    print(
        "The trigram model does not make any predictions for the prompt"
        f" \"{prompt}\" since the bigram \"{' '.join(context_ngram)}\""
        f" is not part of the dataset."
    )

In [41]:
# @title Predict the next token and visualize the distributions

prompt = "Jide was thirsty so she went looking for"  # @param {type: "string"}

output_text_transformer, next_token_logits, tokenizer = (
    generation.prompt_transformer_model(
        prompt, max_new_tokens=1, loaded_model=gemma_model
    )
)

output_text_ngram = trigram_model.generate(1, prompt)

clear_output()

print(f"Generation by Gemma-1B:\n{output_text_transformer}\n\n")
output_text_ngram = trigram_model.generate(1, prompt)

print(f"Generation by trigram model:\n{output_text_ngram}")

display(HTML("<h3>Gemma-1B</h3>"))

# Visualize the Gemma-1B probabilities.
visualizations.plot_next_token(next_token_logits, prompt=prompt, tokenizer=tokenizer)

display(HTML("<h3>Trigram model</h3>"))

# Visualize the trigram probabilities.
context_ngram = tuple(prompt.split(" ")[-2:])
if context_ngram in trigram_model.probabilities:
    visualizations.plot_next_token(
        trigram_model.probabilities[context_ngram], prompt=prompt
    )
else:
    print(
        "The trigram model does not make any predictions for the prompt"
        f" \"{prompt}\ since the bigram \"{' '.join(context_ngram)}\""
        f" is not part of the dataset."
    )

Generation by Gemma-1B:
Jide was thirsty so she went looking for some


Generation by trigram model:
Jide was thirsty so she went looking for Tella,


In [45]:
# @title Generate sequences
prompt = "Jide was hungry so she went looking for"  # @param {type: "string"}

num_tokens_to_generate = 50  # @param {type: "number"}

(output_text_transformer, next_token_logits, tokenizer) = (
    generation.prompt_transformer_model(
        prompt, max_new_tokens=num_tokens_to_generate, loaded_model=gemma_model
    )
)

clear_output()

print(f"Generation by Gemma-1B:\n{output_text_transformer}\n\n")

output_text_ngram = trigram_model.generate(num_tokens_to_generate, prompt)
print(f"Generation by trigram model:\n{output_text_ngram}")

Generation by Gemma-1B:
Jide was hungry so she went looking for some food. She saw the market and decided to grab some chips and drink at a street store. She also told the shopkeeper to make her some chips and drinks so she would have plenty of energy to play with her friends.

She went to


Generation by trigram model:
Jide was hungry so she went looking for a glass of Ginger beer, whose spicy fried plantains, seasoned with spices such as stitching, tying, and applying starch paste, represent proverbs, historical events, and important figures. Worn during special occasions and ceremonies, representing ancestors, spirits, and mythical beings. Dogon sculptures often feature stylized and abstract forms, with elongated figures


In [49]:
# @title Random vs. deterministic generations
prompt = "Jide was thirsty so she went looking for"  # @param {type: "string"}

num_tokens_to_generate = 50  # @param {type: "number"}

sampling_mode = "greedy"  # @param {type: "string", values:["random", "greedy"]}

output_transformer_model = (
    generation.prompt_transformer_model(
        prompt, max_new_tokens = num_tokens_to_generate, sampling_mode=sampling_mode
    )
)

print(f"Generation by Gemma-1B:\n{output_text_transformer}\n\n")

# Visualize the Gemma-1B probabilities.
visualizations.plot_next_token(next_token_logits, prompt=prompt, tokenizer=tokenizer)

Generation by Gemma-1B:
Jide was hungry so she went looking for some food. She saw the market and decided to grab some chips and drink at a street store. She also told the shopkeeper to make her some chips and drinks so she would have plenty of energy to play with her friends.

She went to


